# API truy xuất có bộ lọc metadata

## Mục tiêu tuần này

Thêm các tham số lọc:

- `gender` — giới tính/nhóm người dùng của sản phẩm.
- `category` — danh mục sản phẩm.
- `color` — màu sắc.

vào hai API:

- `POST /api/v1/search/text`
- `POST /api/v1/search/image`

Luồng xử lý:

```text
Text / Image
    ↓
FashionCLIP tạo embedding
    ↓
FAISS tìm nhiều candidate gần nhất
    ↓
Đối chiếu product_id với metadata
    ↓
Lọc gender / category / color
    ↓
Giữ thứ tự similarity
    ↓
Trả Top-K
```

## Quan trọng khi chạy lại sau khi tắt máy

Sau khi **tắt máy / đóng Jupyter / restart kernel**, chạy lại các cell có nhãn:

**[CHẠY LẠI MỖI PHIÊN]**

Các cell cài thư viện có nhãn **[CHỈ CHẠY KHI CẦN]** không cần chạy mỗi lần nếu môi trường Python của máy vẫn giữ nguyên.

Các cell tải model được tách riêng vì đây là phần có thể mất thời gian.

## 1 — Cài thư viện `[CHỈ CHẠY KHI CẦN]`

### Chỉ cài lại khi
- Máy mới.
- Tạo virtual environment mới.
- Báo lỗi `ModuleNotFoundError`.
- Cài lại Python/Jupyter.

Không cần chạy lại chỉ vì tắt máy.

In [ ]:
%pip install -r requirements.txt pandas "transformers==4.46.3"

  Cloning https://github.com/openai/CLIP.git to c:\users\nguyen ho vinh  hien\appdata\local\temp\pip-req-build-yrdv3img
  Resolved https://github.com/openai/CLIP.git to commit d05afc436d78f1c48dc0dbf8e5980a9d471f35f6
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ---------------------------------------- 0.0/10.0 MB ? eta -:--:--
   - -------------------------------------- 0.3/10.0 MB ? eta -:--:--
   --- ------------------------------------ 0.8/10.0 MB 4.8 MB/s eta 0:00:02
   ------- -------------------------------- 1.8/10.0 MB 4.0 MB/s eta 0:00:03
   ------------ --------------------------- 3.1/10.0 MB 4.4 MB/s eta 0:00:02
   -------------- ------------------------- 3.7/10.0 MB 4.4 MB/s eta 0:00:02
   --

  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git 'C:\Users\Nguyen Ho Vinh  Hien\AppData\Local\Temp\pip-req-build-yrdv3img'
  You can safely remove it manually.

[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: C:\Users\Nguyen Ho Vinh  Hien\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## 2 — Xác định thư mục project `[CHẠY LẠI MỖI PHIÊN]`

- Lấy thư mục hiện tại.
- Nếu chưa thấy `embeddings`, thử tìm một thư mục con có `embeddings`.
- Chuyển working directory vào đúng project.

Sau khi tắt máy phải chạy lại vì biến `PROJECT_DIR` trong RAM sẽ mất.

In [3]:
from pathlib import Path
import os

PROJECT_DIR = Path.cwd()

if not (PROJECT_DIR / "embeddings").exists():
    project_candidates = [
        p for p in PROJECT_DIR.iterdir()
        if p.is_dir() and (p / "embeddings").exists()
    ]
    if len(project_candidates) == 1:
        PROJECT_DIR = project_candidates[0]

if not (PROJECT_DIR / "embeddings").exists():
    raise FileNotFoundError(
        "Không tìm thấy thư mục 'embeddings'.\n"
        "Hãy đặt notebook trong thư mục project backend Tuần 36 "
        "hoặc mở Jupyter từ đúng thư mục project."
    )

os.chdir(PROJECT_DIR)

print("PROJECT_DIR =", PROJECT_DIR)
print("Có embeddings:", (PROJECT_DIR / "embeddings").exists())

PROJECT_DIR = c:\Users\Nguyen Ho Vinh  Hien\Downloads\DUAN\fashion_search_backend\fashion_search_backend\week36
Có embeddings: True


## 3 — Import thư viện `[CHẠY LẠI MỖI PHIÊN]`

Nạp các thư viện vào bộ nhớ Python.

Sau khi tắt máy phải chạy lại.

In [1]:
import io
import sqlite3
import threading
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
import torch

from fastapi import FastAPI, File, Form, HTTPException, UploadFile
from fastapi.responses import JSONResponse
from PIL import Image
from pydantic import BaseModel, Field
from transformers import CLIPModel, CLIPProcessor

print("Import thư viện OK")
print("PyTorch device có CUDA ", torch.cuda.is_available())

Import thư viện OK
PyTorch device có CUDA  False


## 4 — Khai báo đường dẫn và cấu hình `[CHẠY LẠI MỖI PHIÊN]`

Khai báo:
- model FashionCLIP,
- đường dẫn FAISS index,
- `product_ids.npy`,
- mẫu URL ảnh,
- vị trí metadata.

Notebook sẽ tự tìm `metadata.csv` hoặc `metadata.db`. Nếu project có nhiều file metadata, có thể gán đường dẫn thủ công vào `METADATA_PATH`.

In [ ]:
MODEL_HF_NAME = "patrickjohncyh/fashion-clip"
MODEL_NAME = "fashionclip"

EMBED_DIR = PROJECT_DIR / "embeddings"
BUILT_INDEX_PATH = EMBED_DIR / f"{MODEL_NAME}_image_faiss.index"
EXISTING_INDEX_PATH = PROJECT_DIR / f"{MODEL_NAME}_image_embeddings_flat.index"
INDEX_PATH = BUILT_INDEX_PATH if BUILT_INDEX_PATH.exists() else EXISTING_INDEX_PATH

PRODUCT_IDS_PATH = EMBED_DIR / "product_ids.npy"
IMAGE_URL_TEMPLATE = "/dataset2/images/{}.jpg"

METADATA_CANDIDATES = [
    PROJECT_DIR / "metadata.csv",
    PROJECT_DIR.parent / "metadata.csv",
    PROJECT_DIR.parent.parent / "metadata.csv",
    PROJECT_DIR.parent.parent.parent / "metadata.csv",
]
METADATA_PATH = next((path for path in METADATA_CANDIDATES if path.exists()), None)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

if not INDEX_PATH.exists():
    raise FileNotFoundError(f"Không tìm thấy FAISS index: {INDEX_PATH}")

if not PRODUCT_IDS_PATH.exists():
    raise FileNotFoundError(f"Không tìm thấy product IDs: {PRODUCT_IDS_PATH}")

print("DEVICE =", DEVICE)
print("INDEX_PATH =", INDEX_PATH)
print("PRODUCT_IDS_PATH =", PRODUCT_IDS_PATH)
print("METADATA_PATH =", METADATA_PATH)

DEVICE = cpu
INDEX_PATH = c:\Users\Nguyen Ho Vinh  Hien\Downloads\DUAN\fashion_search_backend\fashion_search_backend\week36\fashionclip_image_embeddings_flat.index
PRODUCT_IDS_PATH = c:\Users\Nguyen Ho Vinh  Hien\Downloads\DUAN\fashion_search_backend\fashion_search_backend\week36\embeddings\product_ids.npy
METADATA_PATH = c:\Users\Nguyen Ho Vinh  Hien\Downloads\DUAN\metadata.csv


## 5 — Tìm file metadata `[CHẠY LẠI MỖI PHIÊN]`

Tìm `metadata.csv` hoặc `metadata.db` đã tạo ở tuần trước. Ưu tiên `metadata.csv` và `metadata.db` và tìm trong project.

Nếu báo không tìm thấy thì không phải lỗi FashionCLIP/FAISS, chỉ cần chép file metadata vào project, hoặc gán `METADATA_PATH` thủ công ở Cell 4.

In [11]:
def auto_find_metadata(project_dir: Path):
    preferred = [
        project_dir / "metadata.csv",
        project_dir / "metadata.db",
    ]

    for p in preferred:
        if p.exists():
            return p

    csv_candidates = list(project_dir.rglob("metadata.csv"))
    if csv_candidates:
        return csv_candidates[0]

    db_candidates = list(project_dir.rglob("metadata.db"))
    if db_candidates:
        return db_candidates[0]

    return None


if METADATA_PATH is None:
    METADATA_PATH = auto_find_metadata(PROJECT_DIR)

if METADATA_PATH is None:
    raise FileNotFoundError(
        "Không tìm thấy metadata.csv hoặc metadata.db trong project.\n"
        "Hãy copy file metadata của Tuần 35 vào project hoặc gán METADATA_PATH thủ công."
    )

METADATA_PATH = Path(METADATA_PATH)
print("METADATA_PATH =", METADATA_PATH)

METADATA_PATH = c:\Users\Nguyen Ho Vinh  Hien\Downloads\DUAN\metadata.csv


## 6 — Load metadata `[CHẠY LẠI MỖI PHIÊN]`

Đọc bảng metadata có các cột cần cho filter `product_id`, `gender`, `category`, `color`. Sau đó chuẩn hóa tên cột, ép `product_id` thành chuỗi, bỏ trùng theo `product_id`và tạo dictionary để lookup metadata nhanh.

In [12]:
def load_metadata(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()

    if suffix == ".csv":
        df = pd.read_csv(path)

    elif suffix in {".db", ".sqlite", ".sqlite3"}:
        with sqlite3.connect(path) as conn:
            tables = pd.read_sql(
                "SELECT name FROM sqlite_master WHERE type='table';",
                conn
            )["name"].tolist()

            if not tables:
                raise ValueError("SQLite database không có bảng nào.")

            # Ưu tiên tên bảng metadata nếu có.
            table_name = "metadata" if "metadata" in tables else tables[0]
            print("Đang đọc SQLite table:", table_name)
            df = pd.read_sql(f'SELECT * FROM "{table_name}"', conn)
    else:
        raise ValueError(f"Không hỗ trợ file metadata loại: {suffix}")

    # Chuẩn hóa tên cột.
    df.columns = [str(c).strip().lower() for c in df.columns]

    required = {"product_id", "gender", "category", "color"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(
            f"Metadata thiếu cột: {sorted(missing)}\n"
            f"Các cột đang có: {list(df.columns)}"
        )

    df["product_id"] = df["product_id"].astype(str).str.strip()

    for col in ["gender", "category", "color"]:
        df[col] = df[col].fillna("").astype(str).str.strip()

    df = df.drop_duplicates(subset=["product_id"], keep="first").reset_index(drop=True)
    return df


METADATA_DF = load_metadata(METADATA_PATH)

METADATA_BY_ID = (
    METADATA_DF.set_index("product_id")[["gender", "category", "color"]]
    .to_dict(orient="index")
)

print("Metadata rows:", len(METADATA_DF))
print("Metadata lookup entries:", len(METADATA_BY_ID))
display(METADATA_DF.head())

Metadata rows: 44419
Metadata lookup entries: 44419


,product_id,gender,category,color,usage,master_category,sub_category
0,15970,Men,Shirts,Navy Blue,Casual,Apparel,Topwear
1,39386,Men,Jeans,Blue,Casual,Apparel,Bottomwear
2,59263,Women,Watches,Silver,Casual,Accessories,Watches
3,21379,Men,Track Pants,Black,Casual,Apparel,Bottomwear
4,53759,Men,Tshirts,Grey,Casual,Apparel,Topwear


## 7 — Kiểm tra nhanh các giá trị filter `[CHẠY KHI CẦN]`

Xem trong dataset thật đang có các giá trị nào. Không bắt buộc phải chạy mỗi phiên, nhưng nên chạy khi chuẩn bị kiểm thử.

In [13]:
for col in ["gender", "category", "color"]:
    values = sorted(v for v in METADATA_DF[col].dropna().unique() if str(v).strip())
    print(f"\n{col.upper()} — {len(values)} giá trị")
    print(values[:80])
    if len(values) > 80:
        print("...")


GENDER — 5 giá trị
['Boys', 'Girls', 'Men', 'Unisex', 'Women']

CATEGORY — 142 giá trị
['Accessory Gift Set', 'Baby Dolls', 'Backpacks', 'Bangle', 'Basketballs', 'Bath Robe', 'Beauty Accessory', 'Belts', 'Blazers', 'Body Lotion', 'Body Wash And Scrub', 'Booties', 'Boxers', 'Bra', 'Bracelet', 'Briefs', 'Camisoles', 'Capris', 'Caps', 'Casual Shoes', 'Churidar', 'Clothing Set', 'Clutches', 'Compact', 'Concealer', 'Cufflinks', 'Cushion Covers', 'Deodorant', 'Dresses', 'Duffel Bag', 'Dupatta', 'Earrings', 'Eye Cream', 'Eyeshadow', 'Face Moisturisers', 'Face Scrub And Exfoliator', 'Face Serum And Gel', 'Face Wash And Cleanser', 'Flats', 'Flip Flops', 'Footballs', 'Formal Shoes', 'Foundation And Primer', 'Fragrance Gift Set', 'Free Gifts', 'Gloves', 'Hair Accessory', 'Hair Colour', 'Handbags', 'Hat', 'Headband', 'Heels', 'Highlighter And Blush', 'Innerwear Vests', 'Ipad', 'Jackets', 'Jeans', 'Jeggings', 'Jewellery Set', 'Jumpsuit', 'Kajal And Eyeliner', 'Key Chain', 'Kurta Sets', 'Kurtas', '

## 8 — Load `product_ids.npy` và FAISS index `[CHẠY LẠI MỖI PHIÊN]`

Nạp FAISS index vào RAM và danh sách `product_id` dùng để map vị trí vector → sản phẩm.

In [14]:
INDEX = faiss.read_index(str(INDEX_PATH))
PRODUCT_IDS = np.load(PRODUCT_IDS_PATH, allow_pickle=True)

if INDEX.ntotal != len(PRODUCT_IDS):
    raise ValueError(
        f"Index có {INDEX.ntotal} vector nhưng product_ids có "
        f"{len(PRODUCT_IDS)} phần tử."
    )

print(
    f"Loaded {INDEX.ntotal:,} vectors, "
    f"dimension={INDEX.d}, "
    f"product IDs={len(PRODUCT_IDS):,}"
)

Loaded 44,419 vectors, dimension=512, product IDs=44,419


## 9 — Load FashionCLIP Processor `[CHẠY LẠI MỖI PHIÊN]`

`CLIPProcessor` chuẩn bị ảnh/text đúng định dạng đầu vào mà FashionCLIP yêu cầu.

In [15]:
PROCESSOR = CLIPProcessor.from_pretrained(MODEL_HF_NAME)
print("FashionCLIP Processor: OK")

FashionCLIP Processor: OK


## 10 — Load FashionCLIP Model `[CHẠY LẠI MỖI PHIÊN]`

Nạp model FashionCLIP vào RAM/VRAM.

Sau khi tắt máy phải chạy lại vì model trong RAM bị mất.

Lưu ý: Không phải tải lại model từ Internet mỗi lần nếu Hugging Face cache vẫn còn,
nhưng Python vẫn phải **nạp model từ ổ đĩa vào RAM**.

In [16]:
MODEL = CLIPModel.from_pretrained(MODEL_HF_NAME).to(DEVICE)
MODEL.eval()

print("FashionCLIP Model: OK")
print("Đang chạy trên:", DEVICE)

FashionCLIP Model: OK
Đang chạy trên: cpu


## 11 — Hàm tạo embedding `[CHẠY LẠI MỖI PHIÊN]`

Tạo hai hàm `embed_image()`để biến ảnh thành vector và `embed_text()`biến câu mô tả thành vector.

Cell này chỉ khai báo hàm, chưa chạy model trên dữ liệu thật.

In [17]:
def embed_image(pil_image: Image.Image) -> np.ndarray:
    inputs = PROCESSOR(images=pil_image, return_tensors="pt").to(DEVICE)

    with torch.no_grad():
        features = MODEL.get_image_features(**inputs)

    return features.cpu().numpy().astype("float32")


def embed_text(query: str) -> np.ndarray:
    inputs = PROCESSOR(
        text=[query],
        return_tensors="pt",
        padding=True
    ).to(DEVICE)

    with torch.no_grad():
        features = MODEL.get_text_features(**inputs)

    return features.cpu().numpy().astype("float32")


print("Embedding functions: OK")

Embedding functions: OK


## 12 — Hàm chuẩn hóa và kiểm tra filter `[CHẠY LẠI MỖI PHIÊN]`

Cho phép filter không phân biệt chữ hoa/chữ thường và khoảng trắng.

Ví dụ: " men ", "Men", "MEN" đều được xem là cùng một giá trị.

Nếu filter để trống hoặc `None`, điều kiện đó sẽ được bỏ qua.

In [18]:
def normalize_filter_value(value):
    if value is None:
        return None

    value = str(value).strip()
    if value == "":
        return None

    return value.casefold()


def metadata_matches(meta, gender=None, category=None, color=None):
    gender_n = normalize_filter_value(gender)
    category_n = normalize_filter_value(category)
    color_n = normalize_filter_value(color)

    if gender_n is not None:
        if str(meta.get("gender", "")).strip().casefold() != gender_n:
            return False

    if category_n is not None:
        if str(meta.get("category", "")).strip().casefold() != category_n:
            return False

    if color_n is not None:
        if str(meta.get("color", "")).strip().casefold() != color_n:
            return False

    return True


print("Filter functions: OK")

Filter functions: OK


## 13 — Hàm tìm kiếm FAISS + metadata filter `[CHẠY LẠI MỖI PHIÊN]`

Cell này quan trọng nhất vì thực hiện tìm kiếm có bộ lọc metadata. Thay vì chỉ lấy đúng Top-K kết quả từ FAISS rồi mới lọc, hệ thống sẽ lấy trước nhiều sản phẩm có độ tương đồng cao, kiểm tra các điều kiện `gender`, `category` và `color`, sau đó trả về Top-K sản phẩm phù hợp nhất. Nếu số kết quả sau khi lọc chưa đủ, hệ thống sẽ tự động mở rộng số lượng sản phẩm cần kiểm tra cho đến khi đủ kết quả hoặc đã xét toàn bộ dữ liệu.

In [19]:
def search_topk_filtered(
    query_vector: np.ndarray,
    k: int = 10,
    gender: str | None = None,
    category: str | None = None,
    color: str | None = None,
):
    if k < 1:
        raise ValueError("k phải >= 1")

    query_vector = query_vector.astype("float32").reshape(1, -1)
    faiss.normalize_L2(query_vector)

    # Nếu không có filter, giữ cách search đơn giản như Tuần 36.
    no_filter = all(
        normalize_filter_value(v) is None
        for v in [gender, category, color]
    )

    if no_filter:
        candidate_k = min(k, INDEX.ntotal)
    else:
        candidate_k = min(max(k * 20, 200), INDEX.ntotal)

    while True:
        scores, indices = INDEX.search(query_vector, candidate_k)

        results = []

        for score, index in zip(scores[0], indices[0]):
            if index == -1:
                continue

            product_id = str(PRODUCT_IDS[index]).strip()
            meta = METADATA_BY_ID.get(product_id)

            # Sản phẩm không có metadata thì không thể kiểm tra filter.
            if meta is None:
                if no_filter:
                    meta = {"gender": "", "category": "", "color": ""}
                else:
                    continue

            if not metadata_matches(
                meta,
                gender=gender,
                category=category,
                color=color,
            ):
                continue

            results.append(
                {
                    "product_id": product_id,
                    "image_url": IMAGE_URL_TEMPLATE.format(product_id),
                    "similarity_score": round(float(score), 4),
                    "gender": meta.get("gender", ""),
                    "category": meta.get("category", ""),
                    "color": meta.get("color", ""),
                }
            )

            if len(results) >= k:
                return results[:k]

        # Đã xét hết index mà vẫn chưa đủ kết quả.
        if candidate_k >= INDEX.ntotal:
            return results[:k]

        # Mở rộng candidate và search lại.
        candidate_k = min(candidate_k * 2, INDEX.ntotal)


print("Filtered FAISS search: OK")

Filtered FAISS search: OK


## 14 — Sanity check trực tiếp, chưa cần API `[NÊN CHẠY]`

Kiểm tra nhanh model, FAISS, metadata filter trước khi bật FastAPI.

Ví dụ:
- query: `black shirt`
- gender: `Men`
- category: `Shirts`
- color: `Black`

Nếu dataset có sản phẩm phù hợp, tất cả kết quả trả về phải đúng 3 filter.

Cell này dùng để xác định nếu lỗi ở đây thì lỗi thuộc **model / FAISS / metadata / filter**,
không phải FastAPI để debug dễ hơn.

In [20]:
TEST_QUERY = "black shirt"
TEST_GENDER = "Men"
TEST_CATEGORY = "Shirts"
TEST_COLOR = "Black"
TEST_TOP_K = 5

test_vector = embed_text(TEST_QUERY)

test_results = search_topk_filtered(
    test_vector,
    k=TEST_TOP_K,
    gender=TEST_GENDER,
    category=TEST_CATEGORY,
    color=TEST_COLOR,
)

print("Số kết quả:", len(test_results))
display(pd.DataFrame(test_results))

Số kết quả: 5


,product_id,image_url,similarity_score,gender,category,color
0,6048,/dataset2/images/6048.jpg,0.3337,Men,Shirts,Black
1,41603,/dataset2/images/41603.jpg,0.3334,Men,Shirts,Black
2,53264,/dataset2/images/53264.jpg,0.3332,Men,Shirts,Black
3,3009,/dataset2/images/3009.jpg,0.3320,Men,Shirts,Black
4,2050,/dataset2/images/2050.jpg,0.3316,Men,Shirts,Black


## 15 — Tự động kiểm tra kết quả filter `[NÊN CHẠY]`

Cell này dùng `assert` để tự động kiểm tra các kết quả có đúng `gender = Men`, `category = Shirts`, `color = Black` và vẫn được sắp xếp theo `similarity_score` giảm dần hay không. Nếu tất cả điều kiện đều đúng, cell sẽ in `PASS`.

In [21]:
if not test_results:
    print(
        "Không có kết quả cho tổ hợp filter test hiện tại.\n"
        "Hãy xem Cell 7 rồi chọn một tổ hợp gender/category/color có tồn tại trong dataset."
    )
else:
    assert all(
        r["gender"].casefold() == TEST_GENDER.casefold()
        for r in test_results
    ), "FAIL: Có kết quả sai gender"

    assert all(
        r["category"].casefold() == TEST_CATEGORY.casefold()
        for r in test_results
    ), "FAIL: Có kết quả sai category"

    assert all(
        r["color"].casefold() == TEST_COLOR.casefold()
        for r in test_results
    ), "FAIL: Có kết quả sai color"

    scores = [r["similarity_score"] for r in test_results]
    assert scores == sorted(scores, reverse=True), "FAIL: similarity không giảm dần"

    print("PASS — filter và thứ tự similarity đều đúng.")

PASS — filter và thứ tự similarity đều đúng.


## 16 — Khai báo FastAPI `[CHẠY LẠI MỖI PHIÊN]`

Cell này tạo các API tìm kiếm text và image có hỗ trợ bộ lọc `gender`, `category` và `color`. Với text search, dữ liệu được gửi dưới dạng JSON gồm câu truy vấn, số lượng kết quả và các bộ lọc tùy chọn. Với image search, dữ liệu được gửi dưới dạng `multipart/form-data` gồm file ảnh, `top_k` và các bộ lọc. Response trả thêm thông tin metadata của từng sản phẩm để dễ kiểm tra kết quả lọc trên Swagger.

In [22]:
app = FastAPI(title="Fashion Search API - Metadata Filter")

class TextSearchRequest(BaseModel):
    query: str = Field(..., min_length=1)
    top_k: int = Field(10, ge=1, le=50)

    # Optional filters
    gender: str | None = None
    category: str | None = None
    color: str | None = None


@app.exception_handler(HTTPException)
async def http_exception_handler(request, exc):
    return JSONResponse(
        status_code=exc.status_code,
        content={"status": "error", "message": exc.detail},
    )


@app.get("/")
async def health_check():
    return {
        "status": "ok",
        "message": "Fashion Search API + Metadata Filter đang chạy",
    }


@app.post("/search/text")
@app.post("/api/v1/search/text")
async def search_by_text(body: TextSearchRequest):
    results = search_topk_filtered(
        embed_text(body.query),
        k=body.top_k,
        gender=body.gender,
        category=body.category,
        color=body.color,
    )

    return {
        "status": "success",
        "filters": {
            "gender": body.gender,
            "category": body.category,
            "color": body.color,
        },
        "count": len(results),
        "data": results,
    }


@app.post("/search/image")
@app.post("/api/v1/search/image")
async def search_by_image(
    image: UploadFile = File(...),
    top_k: int = Form(10, ge=1, le=50),
    gender: str | None = Form(None),
    category: str | None = Form(None),
    color: str | None = Form(None),
):
    if not image.content_type or not image.content_type.startswith("image/"):
        raise HTTPException(
            status_code=400,
            detail="File phải là ảnh (jpg/png/jpeg)",
        )

    contents = await image.read()

    if not contents:
        raise HTTPException(
            status_code=400,
            detail="File ảnh rỗng",
        )

    try:
        pil_image = Image.open(io.BytesIO(contents)).convert("RGB")
    except Exception as exc:
        raise HTTPException(
            status_code=400,
            detail="Không đọc được ảnh, file có thể bị hỏng",
        ) from exc

    results = search_topk_filtered(
        embed_image(pil_image),
        k=top_k,
        gender=gender,
        category=category,
        color=color,
    )

    return {
        "status": "success",
        "filters": {
            "gender": gender,
            "category": category,
            "color": color,
        },
        "count": len(results),
        "data": results,
    }


print("FastAPI routes: OK")

FastAPI routes: OK


## 17 — Khởi động FastAPI server `[CHẠY LẠI MỖI PHIÊN]`

Chạy Uvicorn ở thread nền để notebook vẫn dùng được.

Swagger:

```text
http://127.0.0.1:8080/docs
```
Sau khi tắt máy phải chạy lại.

Nếu báo port 8080 đang được dùng có thể shutdown server/kernel cũ, hoặc đổi `API_PORT` sang `8000`, `8081`...

In [23]:
import uvicorn

API_HOST = "127.0.0.1"
API_PORT = 8080

if "SERVER_THREAD" not in globals() or not SERVER_THREAD.is_alive():
    SERVER_CONFIG = uvicorn.Config(
        app,
        host=API_HOST,
        port=API_PORT,
        log_level="info",
    )

    SERVER = uvicorn.Server(SERVER_CONFIG)

    SERVER_THREAD = threading.Thread(
        target=SERVER.run,
        daemon=True,
    )

    SERVER_THREAD.start()

    print(f"API đang chạy tại http://{API_HOST}:{API_PORT}/docs")
else:
    print(f"API đã chạy tại http://{API_HOST}:{API_PORT}/docs")

API đang chạy tại http://127.0.0.1:8080/docs


INFO:     Started server process [22076]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8080 (Press CTRL+C to quit)


INFO:     127.0.0.1:53014 - "GET /docs HTTP/1.1" 200 OK
INFO:     127.0.0.1:60477 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     127.0.0.1:60022 - "POST /api/v1/search/text HTTP/1.1" 200 OK
INFO:     127.0.0.1:58824 - "GET /docs HTTP/1.1" 200 OK
INFO:     127.0.0.1:58824 - "GET /openapi.json HTTP/1.1" 200 OK


# HƯỚNG DẪN SỬ DỤNG VÀ KIỂM THỬ API

## 1. Khởi động API

Chạy các cell theo thứ tự đến Cell 17. Khi chạy thành công, API sẽ có tại:

```text
http://127.0.0.1:8080
```

## 2. Mở Swagger

Truy cập:

```text
http://127.0.0.1:8080/docs
```

Trong Swagger có hai endpoint chính:

- `POST /api/v1/search/text` — tìm kiếm bằng mô tả văn bản.
- `POST /api/v1/search/image` — tìm kiếm bằng hình ảnh.

Cả hai endpoint đều hỗ trợ các bộ lọc tùy chọn: `gender`, `category` và `color`.

## 3. Kiểm thử tìm kiếm bằng văn bản

Chọn `POST /api/v1/search/text`, bấm **Try it out** và nhập:

```json
{
  "query": "black shirt",
  "top_k": 5,
  "gender": "Men",
  "category": "Shirts",
  "color": "Black"
}
```

Kết quả trả về gồm `product_id`, `image_url`, `similarity_score` và metadata của từng sản phẩm. Các sản phẩm trả về phải thỏa các bộ lọc đã nhập.

Có thể kiểm thử bằng PowerShell:

```powershell
curl.exe -X POST "http://127.0.0.1:8080/api/v1/search/text" `
  -H "Content-Type: application/json" `
  -d '{"query":"black shirt","top_k":5,"gender":"Men","category":"Shirts","color":"Black"}'
```

## 4. Kiểm thử tìm kiếm bằng hình ảnh

Chọn `POST /api/v1/search/image`, bấm **Try it out**, tải ảnh lên và nhập các trường tùy chọn:

- `top_k`: số lượng kết quả muốn nhận.
- `gender`: giới tính sản phẩm.
- `category`: danh mục sản phẩm.
- `color`: màu sắc sản phẩm.

Có thể kiểm thử bằng PowerShell:

```powershell
curl.exe -X POST "http://127.0.0.1:8080/api/v1/search/image" `
  -F "image=@C:\duong\dan\anh_test.jpg" `
  -F "top_k=5" `
  -F "gender=Men" `
  -F "category=Shirts" `
  -F "color=Black"
```

Nếu không muốn lọc, chỉ cần bỏ qua hoặc để trống các trường `gender`, `category` và `color`. Response vẫn được sắp xếp theo `similarity_score` giảm dần.

# NHỮNG CELL CẦN CHẠY LẠI SAU KHI TẮT MÁY

Sau khi mở notebook lại, chạy theo thứ tự:

```text
Cell 2  — PROJECT_DIR
Cell 3  — Import
Cell 4  — Config/path
Cell 5  — Find metadata
Cell 6  — Load metadata
Cell 8  — Load FAISS + product_ids
Cell 9  — Load Processor
Cell 10 — Load FashionCLIP Model
Cell 11 — Embedding functions
Cell 12 — Filter functions
Cell 13 — Filtered FAISS search
Cell 14 — Sanity check (khuyến nghị)
Cell 15 — Assert test (khuyến nghị)
Cell 16 — FastAPI routes
Cell 17 — Start server
```

Không cần cài thư viện lại nếu môi trường Python vẫn còn đầy đủ.

**Những thứ đã lưu trên ổ đĩa không mất khi tắt máy:**
- FAISS index
- `product_ids.npy`
- `metadata.csv` / `metadata.db`
- Hugging Face cache

**Những thứ trong RAM sẽ mất và phải nạp lại:**
- `MODEL`
- `PROCESSOR`
- `INDEX`
- `PRODUCT_IDS`
- `METADATA_DF`
- `app`
- Uvicorn server